# 01 - Data Profiling: Bairros

Notebook exploratório para análise inicial do dataset `bairros.csv`.

## Objetivos

- conhecer a estrutura dos dados;
- identificar volume e tipos;
- analisar valores nulos e distintos;
- investigar possíveis padrões;
- levantar hipóteses de regras de qualidade.

> Este notebook é exploratório. As observações encontradas aqui
> não representam automaticamente regras de Data Quality.

In [20]:
# 01. Inicialização do Spark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("data-profiling").getOrCreate()

In [25]:
# 02. Definição dos caminhos
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[2]
DATA_PATH = PROJECT_ROOT / "data" / "raw"

print(f"Projeto: {PROJECT_ROOT}")
print(f"Dados:   {DATA_PATH}")

Projeto: /home/claudio/projetos/azure-data-quality-pipeline
Dados:   /home/claudio/projetos/azure-data-quality-pipeline/data/raw


In [27]:
# 03. Carregamento do dataset
bairros = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(str(DATA_PATH / "bairros.csv"))

print("Dataset carregado com sucesso.")

Dataset carregado com sucesso.


In [28]:
# 04. Estrutura do dataset
bairros.printSchema()

root
 |-- codigo: long (nullable = true)
 |-- nome: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- area: double (nullable = true)



## 05. Volume e amostra de dados

Nesta etapa vamos verificar o volume do dataset e observar uma amostra dos registros.

O objetivo é obter uma visão inicial dos dados antes de investigar regras
específicas de qualidade.

In [29]:
print(f"Registros: {bairros.count()}")
print(f"Colunas: {len(bairros.columns)}")

bairros.show(10, truncate=False)

Registros: 133
Colunas: 5
+----------+------------------+-----------+---+--------+
|codigo    |nome              |municipio  |uf |area    |
+----------+------------------+-----------+---+--------+
|355620110 |Observatório      |Valinhos   |SP |68.0009 |
|3519071024|Rp 6-24           |Hortolândia|SP |0.981768|
|3536505002|Jardim De Itapoan |Paulínia   |SP |0.808537|
|3519071026|Rp 6-26           |Hortolândia|SP |2.21108 |
|3536505001|Nova Paulínia     |Paulínia   |SP |0.386199|
|3536505018|Balneário Tropical|Paulínia   |SP |1.70656 |
|3536505019|Nova Veneza       |Paulínia   |SP |1.84108 |
|35095090  |Campo Grande      |Campinas   |SP |47.0085 |
|3536505021|Jardim Itapoan    |Paulínia   |SP |3.1602  |
|3536505024|São Luiz          |Paulínia   |SP |0.12222 |
+----------+------------------+-----------+---+--------+
only showing top 10 rows


## 06. Análise de valores nulos

Nesta etapa vamos verificar a presença de valores nulos em cada coluna.

O objetivo é identificar campos potencialmente incompletos e levantar
hipóteses sobre quais atributos podem ou não aceitar valores nulos.

In [31]:
for coluna in bairros.columns:
    quantidade_nulos = bairros.filter(bairros[coluna].isNull()).count()

    print(f"{coluna}: {quantidade_nulos} nulos")

codigo: 0 nulos
nome: 0 nulos
municipio: 0 nulos
uf: 0 nulos
area: 0 nulos


## 07. Cardinalidade

Nesta etapa vamos analisar a quantidade de valores distintos em cada coluna.

A cardinalidade pode revelar identificadores, atributos com baixa variedade
e possíveis padrões que merecem investigação.

In [32]:
for coluna in bairros.columns:
    quantidade_distintos = bairros.select(coluna).distinct().count()

    print(f"{coluna}: {quantidade_distintos} valores distintos")

codigo: 133 valores distintos
nome: 131 valores distintos
municipio: 7 valores distintos
uf: 1 valores distintos
area: 133 valores distintos


## 08. Investigação de nomes repetidos

A análise de cardinalidade identificou 131 nomes distintos para 133 registros.

Nesta etapa vamos identificar os nomes repetidos e verificar se eles estão
associados ao mesmo município ou a municípios diferentes.

In [33]:
bairros.groupBy("nome") \
    .count() \
    .filter("count > 1") \
    .orderBy("nome") \
    .show(truncate=False)

+------------+-----+
|nome        |count|
+------------+-----+
|Cidade      |2    |
|Observatório|2    |
+------------+-----+



In [34]:
nomes_repetidos = bairros.groupBy("nome") \
    .count() \
    .filter("count > 1") \
    .select("nome")

bairros.join(nomes_repetidos, on="nome", how="inner") \
    .select("codigo", "nome", "municipio", "uf", "area") \
    .orderBy("nome") \
    .show(truncate=False)

+---------+------------+---------+---+-------+
|codigo   |nome        |municipio|uf |area   |
+---------+------------+---------+---+-------+
|355620111|Cidade      |Valinhos |SP |41.5428|
|355670116|Cidade      |Vinhedo  |SP |7.67302|
|355620110|Observatório|Valinhos |SP |68.0009|
|355670114|Observatório|Vinhedo  |SP |14.2438|
+---------+------------+---------+---+-------+



## Observação — Nomes repetidos

Foram identificados 2 nomes repetidos: `Cidade` e `Observatório`.

A análise dos registros mostrou que os nomes repetidos pertencem a municípios
diferentes:

- `Cidade`: Valinhos e Vinhedo;
- `Observatório`: Valinhos e Vinhedo.

Portanto, a repetição do nome não caracteriza, por si só, uma duplicidade
de registro.

### Hipótese

O atributo `nome` não deve ser utilizado isoladamente como identificador
único de um bairro.

## 09. Investigação do código

A coluna `codigo` possui 133 valores distintos para 133 registros.

Nesta etapa vamos investigar a estrutura dos códigos, incluindo seu tamanho,
valores mínimos e máximos e possíveis padrões.

In [35]:
## Estatísticas descritivas do código
bairros.select("codigo") \
    .summary() \
    .show()

+-------+--------------------+
|summary|              codigo|
+-------+--------------------+
|  count|                 133|
|   mean|2.4777340746165414E9|
| stddev|1.5868171702060528E9|
|    min|             3519071|
|    25%|           355620112|
|    50%|          3519071027|
|    75%|          3536505012|
|    max|          3552403006|
+-------+--------------------+



In [36]:
## Investigação do tamanho do código
from pyspark.sql.functions import length

bairros.select(
    "codigo",
    length("codigo").alias("tamanho_codigo")
).orderBy("tamanho_codigo", "codigo") \
 .show(133, truncate=False)

+----------+--------------+
|codigo    |tamanho_codigo|
+----------+--------------+
|3519071   |7             |
|35095064  |8             |
|35095065  |8             |
|35095066  |8             |
|35095067  |8             |
|35095068  |8             |
|35095069  |8             |
|35095070  |8             |
|35095071  |8             |
|35095072  |8             |
|35095073  |8             |
|35095074  |8             |
|35095075  |8             |
|35095076  |8             |
|35095077  |8             |
|35095078  |8             |
|35095079  |8             |
|35095080  |8             |
|35095081  |8             |
|35095082  |8             |
|35095083  |8             |
|35095084  |8             |
|35095085  |8             |
|35095086  |8             |
|35095087  |8             |
|35095088  |8             |
|35095089  |8             |
|35095090  |8             |
|35095091  |8             |
|35095092  |8             |
|35095093  |8             |
|355620110 |9             |
|355620111 |9       

In [37]:
## distribuição do tamanho do código
from pyspark.sql.functions import length

bairros.select(
    length("codigo").alias("tamanho_codigo")
).groupBy("tamanho_codigo") \
 .count() \
 .orderBy("tamanho_codigo") \
 .show()

+--------------+-----+
|tamanho_codigo|count|
+--------------+-----+
|             7|    1|
|             8|   30|
|             9|   10|
|            10|   92|
+--------------+-----+



In [38]:
# distribuição do tamanho do código por município
bairros.groupBy("municipio") \
    .count() \
    .orderBy("municipio") \
    .show()

+-----------+-----+
|  municipio|count|
+-----------+-----+
|   Campinas|   30|
|Hortolândia|   29|
|Nova Odessa|   18|
|   Paulínia|   39|
|     Sumaré|    7|
|   Valinhos|    4|
|    Vinhedo|    6|
+-----------+-----+

